# 01 — Road Network Acquisition (Bangalore + Fallback)
This notebook describes Bangalore OSM acquisition and provides a deterministic fallback graph when internet or OSM tools are unavailable.


## Data provenance
- Preferred source: OpenStreetMap road network extract for a selected Bangalore study area.
- Fallback path: deterministic synthetic directed network (for offline/testing workflows).


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)


In [2]:
# Optional OSM download guidance (not executed automatically):
print("If osmnx is available, you can fetch Bangalore subgraph with drive network type.")
print("This notebook keeps an offline fallback for deterministic execution.")


If osmnx is available, you can fetch Bangalore subgraph with drive network type.
This notebook keeps an offline fallback for deterministic execution.


In [3]:
# Deterministic fallback directed multizone road graph
G = nx.DiGraph()
zones = 12
nodes_per_zone = 15
for z in range(zones):
    for i in range(nodes_per_zone):
        node_id = z * nodes_per_zone + i
        G.add_node(node_id, zone=z)

for z in range(zones):
    base = z * nodes_per_zone
    for i in range(nodes_per_zone - 1):
        u, v = base + i, base + i + 1
        length_m = int(rng.integers(120, 650))
        speed_kmh = int(rng.integers(25, 60))
        G.add_edge(u, v, length_m=length_m, speed_kmh=speed_kmh, oneway=1)
        if rng.random() > 0.2:
            G.add_edge(v, u, length_m=length_m, speed_kmh=speed_kmh, oneway=0)

for z in range(zones - 1):
    a = z * nodes_per_zone + int(rng.integers(0, nodes_per_zone))
    b = (z + 1) * nodes_per_zone + int(rng.integers(0, nodes_per_zone))
    G.add_edge(a, b, length_m=int(rng.integers(200, 900)), speed_kmh=int(rng.integers(30, 55)), oneway=1)

print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
print("Weakly connected:", nx.is_weakly_connected(G))


Nodes: 180 Edges: 313
Weakly connected: True


In [4]:
edges = []
for u, v, d in G.edges(data=True):
    edges.append({"u":u,"v":v,"length_m":d["length_m"],"speed_kmh":d["speed_kmh"],"oneway":d["oneway"],"source":"deterministic_fallback"})

edge_df = pd.DataFrame(edges)
edge_path = DATA_DIR / "road_edges_fallback.csv"
edge_df.to_csv(edge_path, index=False)
print("Saved:", edge_path, "rows=", len(edge_df))


Saved: /home/runner/work/Q-Optima/Q-Optima/gat-traffic-prediction/data/raw/road_edges_fallback.csv rows= 313


## Notes
- Historical traffic measurements are separate from topology.
- This fallback graph is only for pipeline development, not a claim about real Bangalore traffic.
